AutoMed is a multi-agent AI healthcare assistant built with AG2 (AutoGen), where multiple specialized AI agents collaborate — a triage agent, a diagnosis agent, and a treatment recommender — to simulate a real medical consultation flow.

In [1]:
!pip install ag2 pyautogen

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 13.3 MB/s  0:00:00

  Attempting uninstall: protobuf

    Found existing installation: protobuf 6.33.6

    Uninstalling protobuf-6.33.6:

      Successfully uninstalled protobuf-6.33.6

   ----- ---------------------------------- 1/7 [protobuf]
   ----- ---------------------------------- 1/7 [protobuf]
   ----- ---------------------------------- 1/7 [protobuf]
   ----- ---------------------------------- 1/7 [protobuf]
   ----- ---------------------------------- 1/7 [protobuf]
   ----- ---------------------------------- 1/7 [protobuf]
   ----- ---------------------------------- 1/7 [protobuf]
   ----- ---------------------------------- 1/7 [protobuf]
   ----- ---------------------------------- 1/7 [protobuf]
   ----- ---------------------------------- 1/7 [protobuf]
   ----- ---------------------------------- 1/7 [protobuf]
   ----- -------------------------------

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
logfire 4.33.0 requires opentelemetry-sdk<1.42.0,>=1.39.0, but you have opentelemetry-sdk 1.42.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.39.1 requires opentelemetry-exporter-otlp-proto-common==1.39.1, but you have opentelemetry-exporter-otlp-proto-common 1.42.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.39.1 requires opentelemetry-proto==1.39.1, but you have opentelemetry-proto 1.42.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.39.1 requires opentelemetry-sdk~=1.39.1, but you have opentelemetry-sdk 1.42.1 which is incompatible.


Setting Up configuration:

In [15]:
import requests

OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL_NAME = "llama3.2:latest"

llm_config = {
    "config_list": [
        {
            "model": MODEL_NAME,
            "base_url": OLLAMA_BASE_URL,
            "api_key": "ollama",
        }
    ],
    "temperature": 0.3,
    "cache_seed": None,
}

# Quick sanity check — pings Ollama to confirm the model is reachable
try:
    response = requests.post(
        "http://localhost:11434/v1/chat/completions",
        json={
            "model": MODEL_NAME,
            "messages": [{"role": "user", "content": "Reply with just: OK"}],
        },
        timeout=30,
    )
    reply = response.json()["choices"][0]["message"]["content"]
    print(f"✅ Ollama connected. Model: {MODEL_NAME}")
    print(f"   Test reply: {reply.strip()}")
except Exception as e:
    print(f"❌ Connection failed: {e}")

✅ Ollama connected. Model: llama3.2:latest
   Test reply: OK


Triage Agent (Cell 3):
1.collects symptoms
2.assesses urgency

In [16]:
import autogen

triage_agent = autogen.ConversableAgent(
    name="TriageAgent",
    system_message="""You are a medical triage specialist.
Your job is to:
1. Acknowledge the patient's complaint.
2. Note the chief symptom, duration, severity (1-10), and any associated symptoms.
3. Assess urgency: Emergency / Urgent / Non-urgent.
4. Summarize findings clearly for the Diagnosis Agent.

NEVER give a diagnosis yourself. Only collect info and assess urgency.
Always end your message with: TRIAGE COMPLETE: [urgency level]""",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

print("Triage Agent ready.")

Triage Agent ready.


Diagnosis Agent:
Maps symptoms to condition

In [17]:
diagnosis_agent = autogen.ConversableAgent(
    name="DiagnosisAgent",
    system_message="""You are an experienced medical diagnostician.
You receive triage summaries and:
1. Analyze the symptom pattern.
2. List 2-3 possible differential diagnoses (most to least likely).
3. Briefly explain the reasoning for each.
4. Flag any red-flag symptoms needing immediate attention.
5. State your confidence: High / Medium / Low.

Format your output as:
POSSIBLE CONDITIONS:
1. [Condition] — [Reasoning] — Confidence: [level]
2. ...

End with: DIAGNOSIS COMPLETE: proceed to treatment""",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

print("Diagnosis Agent ready.")

Diagnosis Agent ready.


Treatment Agent:
Recommends care and medications

In [18]:
treatment_agent = autogen.ConversableAgent(
    name="TreatmentAgent",
    system_message="""You are a clinical treatment advisor.
Based on the diagnosis provided:
1. Recommend first-line treatments (lifestyle changes, OTC medications if safe).
2. Specify clearly when the patient MUST see a doctor in person.
3. List medications or activities to AVOID.
4. Give simple home care advice.

Always end with this exact disclaimer:
DISCLAIMER: This is AI-generated guidance only. Always consult a licensed
physician for diagnosis and treatment decisions.""",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

print("Treatment Agent ready.")

Treatment Agent ready.


UserProxy + GroupChat + Manager:

In [19]:
# UserProxyAgent represents the patient — it feeds the initial complaint in
user_proxy = autogen.UserProxyAgent(
    name="Patient",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=0,  # patient speaks once, then agents take over
    code_execution_config=False,   # no code execution needed in a medical chat
)

# GroupChat is the shared "room" all agents read from and write to
group_chat = autogen.GroupChat(
    agents=[user_proxy, triage_agent, diagnosis_agent, treatment_agent],
    messages=[],
    max_round=6,
    speaker_selection_method="round_robin",  # triage → diagnosis → treatment in order
)

# GroupChatManager drives who speaks next
manager = autogen.GroupChatManager(
    groupchat=group_chat,
    llm_config=llm_config,
)

print("GroupChat and Manager ready.")

GroupChat and Manager ready.


In [14]:
import ollama

models = ollama.list()
print(models)


models=[Model(model='nomic-embed-text:latest', modified_at=datetime.datetime(2026, 5, 27, 13, 13, 23, 31061, tzinfo=TzInfo(19800)), digest='0a109f422b47e3a30ba2b10eca18548e944e8a23073ee3f3e947efcf3c45e59f', size=274302450, details=ModelDetails(parent_model='', format='gguf', family='nomic-bert', families=['nomic-bert'], parameter_size='137M', quantization_level='F16')), Model(model='llama3.1:8b', modified_at=datetime.datetime(2026, 5, 27, 12, 58, 23, 175641, tzinfo=TzInfo(19800)), digest='46e0c10c039e019119339687c3c1757cc81b9da49709a3b3924863ba87ca666e', size=4920753328, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], parameter_size='8.0B', quantization_level='Q4_K_M')), Model(model='llava:latest', modified_at=datetime.datetime(2026, 5, 20, 17, 48, 2, 43207, tzinfo=TzInfo(19800)), digest='8dd30f6b0cb19f555f2c7a7ebda861449ea2cc76bf1f44e262931f45fc81d081', size=4733363377, details=ModelDetails(parent_model='', format='gguf', family='llama', famili

In [20]:
# Change this to any symptom description you want to test
patient_input = "I have a severe headache behind my eyes, fever of 38.5°C, and stiff neck since this morning."

print("=" * 60)
print("  AutoMed — AI Healthcare Assistant")
print("  Running on local Ollama —", MODEL_NAME)
print("=" * 60)
print(f"\nPatient: {patient_input}\n")

user_proxy.initiate_chat(
    manager,
    message=f"Patient complaint: {patient_input}",
)

  AutoMed — AI Healthcare Assistant
  Running on local Ollama — llama3.2:latest

Patient: I have a severe headache behind my eyes, fever of 38.5°C, and stiff neck since this morning.

Patient (to chat_manager):

Patient complaint: I have a severe headache behind my eyes, fever of 38.5°C, and stiff neck since this morning.

--------------------------------------------------------------------------------

Next speaker: TriageAgent

[autogen.oai.client: 05-27 14:22:07] {734} WARNING - Model llama3.2:latest is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
TriageAgent (to chat_manager):

Here's the patient information:

* Chief symptom: Severe headache behind the eyes
* Duration: Since this morning
* Severity (1-10): 8
* Associated symptoms: Fever of 38.5°C, stiff neck
* Urgency assessment: Emergency

TRIAGE COMPLETE: EMERGENCY

-----------------------------------------------------------

ChatResult(chat_id=101842688167883982691358073110365167489, chat_history=[{'content': 'Patient complaint: I have a severe headache behind my eyes, fever of 38.5°C, and stiff neck since this morning.', 'role': 'assistant', 'name': 'Patient'}, {'content': "Here's the patient information:\n\n* Chief symptom: Severe headache behind the eyes\n* Duration: Since this morning\n* Severity (1-10): 8\n* Associated symptoms: Fever of 38.5°C, stiff neck\n* Urgency assessment: Emergency\n\nTRIAGE COMPLETE: EMERGENCY", 'name': 'TriageAgent', 'role': 'user'}, {'content': 'POSSIBLE CONDITIONS:\n1. Meningitis — The combination of severe headache behind the eyes, fever, and stiff neck suggests a possible infection of the meninges, which are the protective membranes surrounding the brain and spinal cord. This is a red-flag symptom for meningitis, especially given the emergency urgency assessment. Confidence: High\n2. Subarachnoid hemorrhage — The sudden onset of severe headache behind the eyes could indic